# LoRA Operations Notebook

Interactive notebook for managing LoRA adapter lifecycle:
register, promote, demote, download, and sync adapters.

## Sections
1. Setup & Configuration
2. Inspect Training Runs
3. Register Adapter
4. Promote / Demote
5. Sync to Inference Host

## 1. Setup & Configuration

In [ ]:
import importlib
import os
import sys
from pathlib import Path

from IPython.display import display

PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", Path("../..").resolve())).resolve()
os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)

for p in [str(PROJECT_ROOT), str(PROJECT_ROOT / "src")]:
    if p not in sys.path:
        sys.path.insert(0, p)

from shared.local_env import load_local_env

load_local_env(repo_root=PROJECT_ROOT)

mlflow = importlib.import_module("mlflow")

from shared.model_registry import AdapterRegistry, AdapterSyncer

tracking_uri = os.getenv("MLFLOW_TRACKING_URI")
if tracking_uri:
    mlflow.set_tracking_uri(tracking_uri)

registry = AdapterRegistry()

print(f"PROJECT_ROOT     : {PROJECT_ROOT}")
print(f"MLflow tracking  : {mlflow.get_tracking_uri()}")

## 2. Inspect Training Runs

List recent MLflow runs together with key training, lineage, and post-train evaluation metrics. Use this table before deciding whether a run is worth registering.

In [ ]:
EXPERIMENT_NAME = "train_adapter"

runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    max_results=10,
    order_by=["start_time DESC"],
)

if runs.empty:
    print(f"No runs found in experiment '{EXPERIMENT_NAME}'.")
else:
    metric_columns = sorted(c for c in runs.columns if c.startswith("metrics.eval."))
    cols = [
        c
        for c in [
            "run_id",
            "status",
            "start_time",
            "metrics.val_loss",
            "metrics.train_loss_epoch",
            "metrics.zero_target_ratio_epoch",
            "params.dataset_dvc_hash",
            "params.effective_batch_size",
            "params.trainable_param_count",
            "tags.git.sha",
            *metric_columns,
        ]
        if c in runs.columns
    ]
    display(runs[cols])

## 3. Register Adapter

Register a trained adapter from an MLflow run into the Model Registry.

In [ ]:
# Replace with values from the table above
RUN_ID = ""
MODEL_NAME = "lora-summarize"
ARTIFACT_PATH = "model"

if RUN_ID:
    mv = registry.register_adapter(
        run_id=RUN_ID,
        artifact_path=ARTIFACT_PATH,
        model_name=MODEL_NAME,
    )
    print(f"Registered {MODEL_NAME} v{mv.version} from run {RUN_ID}")
else:
    print("Set RUN_ID above to register an adapter.")

## 4. Promote / Demote

Assign or remove aliases (champion, challenger) on registered versions.

In [ ]:
MODEL_NAME = "lora-summarize"
VERSION = None  # set to an integer to promote an alias
ALIAS = "champion"
REMOVE_ALIAS = False

if REMOVE_ALIAS:
    registry.demote(model_name=MODEL_NAME, alias=ALIAS)
    print(f"Removed alias '{ALIAS}' from '{MODEL_NAME}'")
elif VERSION is not None:
    registry.promote(model_name=MODEL_NAME, version=int(VERSION), alias=ALIAS)
    print(f"Promoted '{MODEL_NAME}' v{VERSION} to alias '{ALIAS}'")
else:
    print("Set VERSION to promote, or set REMOVE_ALIAS=True to demote.")

versions = registry.list_versions(model_name=MODEL_NAME)
if versions:
    print("\nRegistered versions:")
    for v in versions:
        print(f"  v{v.version}  aliases={v.aliases}  run_id={v.run_id}")
else:
    print(f"No registered versions for '{MODEL_NAME}'.")

## 5. Sync to Inference Host

Download champion adapters and hot-load them into a running vLLM instance.

In [ ]:
ADAPTERS_DIR = PROJECT_ROOT / "assets" / "adapters"
VLLM_URL = ""  # e.g. http://localhost:8000
SYNC_ALIASES = ["champion", "challenger"]

if VLLM_URL:
    syncer = AdapterSyncer(
        adapters_dir=ADAPTERS_DIR,
        sync_aliases=SYNC_ALIASES,
        vllm_base_url=VLLM_URL,
    )
    infos = syncer.sync()
    if infos:
        print(f"Synced {len(infos)} adapter(s):")
        for info in infos:
            print(f"  {info.name}  v{info.version}  ->  {info.local_path}")
    else:
        print("No adapters were synced.")
else:
    print("Set VLLM_URL above to sync adapters into a running vLLM instance.")